In [1]:
#!/usr/bin/env python3
"""
═══════════════════════════════════════════════════════════════════════════════
TOKENIZED REASONING NETWORK (TRN) FOR VISUAL REINFORCEMENT LEARNING
═══════════════════════════════════════════════════════════════════════════════

Novel Architecture:
  1. Perception Tokenizer: CNN → Spatial Tokens with Positional Embeddings
  2. Cross-Attention Translator: Actions QUERY perception tokens
  3. Action Token Decoding: Magnitude-based selection with bias
  4. Contrastive Loss: Strengthens action-perception alignment

Key Insight: Actions are not output indices but semantic queries that
             attend to perception, producing rich action representations.

Environment: MinAtar Breakout (single game for clean validation)
Compute: Kaggle P100 16GB
Seeds: 42, 123, 456
═══════════════════════════════════════════════════════════════════════════════
"""

import subprocess
import sys

def install_packages():
    pkgs = ['gymnasium', 'minatar', 'matplotlib', 'seaborn']
    for pkg in pkgs:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        except:
            pass

print("=" * 80)
print(" TOKENIZED REASONING NETWORK (TRN) - INSTALLATION")
print("=" * 80)
install_packages()

import os
import time
import random
import math
import warnings
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy import stats

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
sns.set_palette("husl")

warnings.filterwarnings('ignore')

MINATAR_AVAILABLE = False
try:
    import minatar
    MINATAR_AVAILABLE = True
    print("MinAtar: Available")
except ImportError:
    print("MinAtar: Not available, using fallback")

import gymnasium as gym
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    seeds: List[int] = field(default_factory=lambda: [42, 123, 456])
    game: str = 'breakout'  # Single game only
    
    obs_channels: int = 10
    num_actions: int = 6
    
    # Embedding dimensions
    embed_dim: int = 64
    num_perception_tokens: int = 25  # 5x5 spatial grid
    
    # Training
    learning_rate: float = 3e-4
    num_train_steps: int = 300  # Reduced for speed
    num_steps_per_rollout: int = 256
    num_envs: int = 8
    
    # PPO
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_ratio: float = 0.2
    value_coef: float = 0.5
    entropy_coef: float = 0.15
    max_grad_norm: float = 0.5
    ppo_epochs: int = 3
    
    # Novel components
    contrastive_weight: float = 0.1
    contrastive_margin: float = 0.3
    
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./trn_figures"

config = Config()
os.makedirs(config.save_dir, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def print_section(title: str):
    print("\n" + "=" * 80)
    print(f" {title}")
    print("=" * 80)

def print_subsection(title: str):
    print("\n" + "-" * 60)
    print(f" {title}")
    print("-" * 60)

# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1: ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════════════════════

class MinAtarEnv:
    def __init__(self, game: str = 'breakout'):
        self.game = game
        self.env = minatar.Environment(game)
        self.action_dim = self.env.num_actions()
        
    def reset(self) -> np.ndarray:
        self.env.reset()
        return self._get_obs()
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        reward, done = self.env.act(action)
        return self._get_obs(), float(reward), done, {}
    
    def _get_obs(self) -> np.ndarray:
        state = self.env.state().astype(np.float32)
        obs = np.transpose(state, (2, 0, 1))
        if obs.shape[0] < 10:
            pad = np.zeros((10 - obs.shape[0], 10, 10), dtype=np.float32)
            obs = np.concatenate([obs, pad], axis=0)
        return obs
    
    def close(self):
        pass

class FallbackEnv:
    def __init__(self):
        self.env = gym.make('CartPole-v1')
        self.action_dim = 2
        
    def reset(self) -> np.ndarray:
        obs, _ = self.env.reset()
        return self._to_visual(obs)
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        obs, reward, term, trunc, info = self.env.step(int(action) % self.action_dim)
        return self._to_visual(obs), float(reward), term or trunc, info
    
    def _to_visual(self, state: np.ndarray) -> np.ndarray:
        visual = np.zeros((10, 10, 10), dtype=np.float32)
        state = np.array(state).flatten()
        for i, val in enumerate(state[:4]):
            x = int(np.clip((val + 3) / 6.0, 0, 1) * 9)
            visual[i % 10, :, x] = 1.0
        return visual
    
    def close(self):
        self.env.close()

class VecEnv:
    def __init__(self, game: str, num_envs: int):
        self.num_envs = num_envs
        self.game = game
        if MINATAR_AVAILABLE:
            self.envs = [MinAtarEnv(game) for _ in range(num_envs)]
        else:
            self.envs = [FallbackEnv() for _ in range(num_envs)]
        self.action_dim = self.envs[0].action_dim
        
    def reset(self) -> np.ndarray:
        return np.stack([e.reset() for e in self.envs])
    
    def step(self, actions: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List]:
        results = []
        for i, env in enumerate(self.envs):
            obs, r, done, info = env.step(int(actions[i]) % env.action_dim)
            if done:
                obs = env.reset()
            results.append((obs, r, done, info))
        return (np.stack([r[0] for r in results]),
                np.array([r[1] for r in results]),
                np.array([r[2] for r in results]),
                [r[3] for r in results])
    
    def close(self):
        for e in self.envs:
            e.close()

# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def save_learning_curve(rewards: List[float], title: str, filepath: str, 
                        baseline_rewards: List[float] = None):
    """Save learning curve plot immediately after training."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Smooth rewards
    window = min(20, len(rewards) // 5) if len(rewards) > 20 else 1
    if window > 1:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(smoothed, 'b-', linewidth=2, label=f'{title} (smoothed)')
        ax.plot(rewards, 'b-', alpha=0.2, linewidth=1)
    else:
        ax.plot(rewards, 'b-', linewidth=2, label=title)
    
    if baseline_rewards is not None:
        if window > 1:
            baseline_smooth = np.convolve(baseline_rewards, np.ones(window)/window, mode='valid')
            ax.plot(baseline_smooth, 'r-', linewidth=2, label='Baseline (smoothed)')
            ax.plot(baseline_rewards, 'r-', alpha=0.2, linewidth=1)
        else:
            ax.plot(baseline_rewards, 'r-', linewidth=2, label='Baseline')
    
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Mean Episode Reward')
    ax.set_title(f'Learning Curve: {title}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {filepath}")


def save_attention_heatmap(attention_weights: np.ndarray, filepath: str):
    """Visualize cross-attention weights: which perception tokens each action attends to."""
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    action_names = ['Action 0', 'Action 1', 'Action 2', 'Action 3', 'Action 4', 'Action 5']
    
    for i, ax in enumerate(axes.flat):
        if i < attention_weights.shape[0]:
            # Reshape to 5x5 spatial grid
            attn = attention_weights[i].reshape(5, 5)
            im = ax.imshow(attn, cmap='hot', vmin=0, vmax=attention_weights.max())
            ax.set_title(f'{action_names[i]}')
            ax.set_xlabel('X position')
            ax.set_ylabel('Y position')
            plt.colorbar(im, ax=ax, fraction=0.046)
        else:
            ax.axis('off')
    
    plt.suptitle('Cross-Attention: Where Each Action Looks', fontsize=14)
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {filepath}")


def save_action_token_magnitudes(magnitudes_history: List[np.ndarray], filepath: str):
    """Plot how action token magnitudes evolve during training."""
    if not magnitudes_history:
        return
    
    magnitudes = np.array(magnitudes_history)  # (steps, 6)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i in range(magnitudes.shape[1]):
        ax.plot(magnitudes[:, i], label=f'Action {i}', linewidth=2, alpha=0.8)
    
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Action Token Magnitude')
    ax.set_title('Action Token Activation During Training')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {filepath}")


def save_perception_tokens_pca(perception_tokens: np.ndarray, filepath: str):
    """Visualize perception tokens in 2D via PCA."""
    if perception_tokens.shape[0] < 10:
        return
    
    # Flatten batch and tokens
    tokens_flat = perception_tokens.reshape(-1, perception_tokens.shape[-1])
    
    if tokens_flat.shape[0] > 2:
        pca = PCA(n_components=2)
        proj = pca.fit_transform(tokens_flat)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Color by spatial position
        positions = np.tile(np.arange(25), perception_tokens.shape[0])
        scatter = ax.scatter(proj[:, 0], proj[:, 1], c=positions, cmap='viridis', 
                            alpha=0.5, s=20)
        plt.colorbar(scatter, ax=ax, label='Spatial Position (0-24)')
        
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        ax.set_title('Perception Tokens in PCA Space')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {filepath}")


def save_comparison_bar(results: Dict, filepath: str):
    """Bar chart comparing methods."""
    methods = list(results.keys())
    means = [results[m]['mean'] for m in methods]
    stds = [results[m]['std'] for m in methods]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(methods)))
    bars = ax.bar(range(len(methods)), means, yerr=stds, capsize=8, 
                  color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=15, ha='right')
    ax.set_ylabel('Mean Reward')
    ax.set_title('Performance Comparison')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.1,
                f'{mean:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {filepath}")


def save_ablation_chart(ablation_results: Dict, filepath: str):
    """Horizontal bar chart for ablation study."""
    components = list(ablation_results.keys())
    values = [ablation_results[c]['mean'] for c in components]
    
    # Sort by value
    sorted_idx = np.argsort(values)[::-1]
    components = [components[i] for i in sorted_idx]
    values = [values[i] for i in sorted_idx]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(components)))
    bars = ax.barh(range(len(components)), values, color=colors, edgecolor='black')
    
    ax.set_yticks(range(len(components)))
    ax.set_yticklabels(components)
    ax.set_xlabel('Mean Reward')
    ax.set_title('Component Ablation Study')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', ha='left', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {filepath}")


def save_dimension_comparison(dim_results: Dict, filepath: str):
    """Compare different embedding dimensions."""
    dims = sorted(dim_results.keys())
    means = [dim_results[d]['mean'] for d in dims]
    stds = [dim_results[d]['std'] for d in dims]
    params = [dim_results[d]['params'] for d in dims]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Performance
    ax = axes[0]
    bars = ax.bar(range(len(dims)), means, yerr=stds, capsize=8,
                  color='steelblue', edgecolor='black')
    ax.set_xticks(range(len(dims)))
    ax.set_xticklabels([f'd={d}' for d in dims])
    ax.set_xlabel('Embedding Dimension')
    ax.set_ylabel('Mean Reward')
    ax.set_title('Performance vs Dimension')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{mean:.2f}', ha='center', va='bottom', fontsize=10)
    
    # Parameters
    ax = axes[1]
    ax.bar(range(len(dims)), params, color='coral', edgecolor='black')
    ax.set_xticks(range(len(dims)))
    ax.set_xticklabels([f'd={d}' for d in dims])
    ax.set_xlabel('Embedding Dimension')
    ax.set_ylabel('Parameters')
    ax.set_title('Model Size vs Dimension')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {filepath}")


# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2: BASELINE CNN-PPO
# ═══════════════════════════════════════════════════════════════════════════════

class BaselineCNN(nn.Module):
    """Standard CNN-PPO baseline with NO novel components."""
    def __init__(self, num_actions: int = 6, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim
        
        self.encoder = nn.Sequential(
            nn.Conv2d(10, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 3 * 3, embed_dim),
            nn.ReLU()
        )
        
        self.actor = nn.Linear(embed_dim, num_actions)
        self.critic = nn.Linear(embed_dim, 1)
        
    def forward(self, obs):
        feat = self.encoder(obs)
        return self.actor(feat), self.critic(feat).squeeze(-1)
    
    def get_action_and_value(self, obs, action=None):
        logits, value = self.forward(obs)
        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), value, dist.entropy(), None, None
    
    def get_value(self, obs):
        _, value = self.forward(obs)
        return value
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ═══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3: TOKENIZED REASONING NETWORK (TRN)
# ═══════════════════════════════════════════════════════════════════════════════

class PerceptionTokenizer(nn.Module):
    """
    Stage 1: Convert visual input to perception tokens with positional embeddings.
    CNN extracts features → reshape to spatial tokens → add 2D positions.
    """
    def __init__(self, in_channels: int = 10, embed_dim: int = 64):
        super().__init__()
        
        # CNN feature extraction
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, embed_dim, 3, stride=2, padding=1),
            nn.ReLU()
        )  # Output: (B, embed_dim, 5, 5)
        
        # 2D positional embeddings for 5x5 grid
        self.pos_embed = nn.Parameter(torch.randn(25, embed_dim) * 0.02)
        
        # Initialize positions meaningfully
        with torch.no_grad():
            for i in range(5):
                for j in range(5):
                    idx = i * 5 + j
                    # Encode x, y position in first dimensions
                    self.pos_embed[idx, 0] = i / 4.0 - 0.5  # x: -0.5 to 0.5
                    self.pos_embed[idx, 1] = j / 4.0 - 0.5  # y: -0.5 to 0.5
        
    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        """
        obs: (B, C, 10, 10)
        returns: (B, 25, embed_dim) perception tokens
        """
        features = self.cnn(obs)  # (B, embed_dim, 5, 5)
        B, C, H, W = features.shape
        
        # Reshape to tokens
        tokens = features.view(B, C, H * W).permute(0, 2, 1)  # (B, 25, embed_dim)
        
        # Add positional embeddings
        tokens = tokens + self.pos_embed.unsqueeze(0)
        
        return tokens


class CrossAttentionTranslator(nn.Module):
    """
    Stage 2: Actions QUERY perception tokens via cross-attention.
    Each action learns what perceptual information it needs.
    """
    def __init__(self, num_actions: int = 6, embed_dim: int = 64):
        super().__init__()
        self.num_actions = num_actions
        self.embed_dim = embed_dim
        self.scale = embed_dim ** -0.5
        
        # Learnable action queries - what each action "asks for"
        self.action_queries = nn.Parameter(torch.randn(num_actions, embed_dim) * 0.02)
        
        # Projection layers for attention
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, perception_tokens: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        perception_tokens: (B, 25, embed_dim)
        returns: 
            action_tokens: (B, num_actions, embed_dim)
            attention_weights: (B, num_actions, 25)
        """
        B = perception_tokens.shape[0]
        
        # Expand action queries for batch
        queries = self.action_queries.unsqueeze(0).expand(B, -1, -1)  # (B, 6, D)
        
        # Project
        Q = self.q_proj(queries)        # (B, 6, D)
        K = self.k_proj(perception_tokens)  # (B, 25, D)
        V = self.v_proj(perception_tokens)  # (B, 25, D)
        
        # Attention scores
        attn_scores = torch.bmm(Q, K.transpose(-2, -1)) * self.scale  # (B, 6, 25)
        attn_weights = F.softmax(attn_scores, dim=-1)  # (B, 6, 25)
        
        # Attend to values
        action_tokens = torch.bmm(attn_weights, V)  # (B, 6, D)
        action_tokens = self.out_proj(action_tokens)
        
        return action_tokens, attn_weights


class ActionDecoder(nn.Module):
    """
    Stage 3: Decode action tokens to logits via magnitude + bias.
    """
    def __init__(self, num_actions: int = 6, embed_dim: int = 64):
        super().__init__()
        
        # Action bias (baseline preferences)
        self.action_bias = nn.Parameter(torch.zeros(num_actions))
        
        # Optional: learned scaling per action
        self.action_scale = nn.Parameter(torch.ones(num_actions))
        
    def forward(self, action_tokens: torch.Tensor) -> torch.Tensor:
        """
        action_tokens: (B, num_actions, embed_dim)
        returns: logits (B, num_actions)
        """
        # Magnitude of each action token
        magnitudes = torch.norm(action_tokens, dim=-1)  # (B, num_actions)
        
        # Scale and add bias
        logits = magnitudes * self.action_scale + self.action_bias
        
        return logits, magnitudes


class TokenizedReasoningNetwork(nn.Module):
    """
    Full TRN: Perception Tokenizer → Cross-Attention Translator → Action Decoder
    
    Novel Components:
    1. Spatial tokens with positional embeddings
    2. Action queries that attend to perception
    3. Magnitude-based action decoding
    4. Contrastive loss for action separation
    """
    def __init__(self, num_actions: int = 6, embed_dim: int = 64,
                 use_cross_attention: bool = True,
                 use_positional: bool = True,
                 use_contrastive: bool = True,
                 use_bias: bool = True):
        super().__init__()
        self.num_actions = num_actions
        self.embed_dim = embed_dim
        self.use_cross_attention = use_cross_attention
        self.use_positional = use_positional
        self.use_contrastive = use_contrastive
        self.use_bias = use_bias
        
        # Stage 1: Perception Tokenizer
        self.tokenizer = PerceptionTokenizer(10, embed_dim)
        if not use_positional:
            self.tokenizer.pos_embed.requires_grad = False
            self.tokenizer.pos_embed.zero_()
        
        # Stage 2: Cross-Attention Translator
        if use_cross_attention:
            self.translator = CrossAttentionTranslator(num_actions, embed_dim)
        else:
            # Ablation: simple pooling + linear
            self.pool_proj = nn.Linear(embed_dim, embed_dim * num_actions)
        
        # Stage 3: Action Decoder
        self.decoder = ActionDecoder(num_actions, embed_dim)
        if not use_bias:
            self.decoder.action_bias.requires_grad = False
            self.decoder.action_bias.zero_()
        
        # Value head (from pooled perception)
        self.value_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1)
        )
        
        # Storage for visualization
        self.last_attention = None
        self.last_magnitudes = None
        self.last_perception_tokens = None
        
    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        obs: (B, C, 10, 10)
        returns: logits, value, magnitudes
        """
        B = obs.shape[0]
        
        # Stage 1: Tokenize perception
        perception_tokens = self.tokenizer(obs)  # (B, 25, D)
        self.last_perception_tokens = perception_tokens.detach()
        
        # Stage 2: Translate to action tokens
        if self.use_cross_attention:
            action_tokens, attn_weights = self.translator(perception_tokens)
            self.last_attention = attn_weights.detach()
        else:
            # Ablation: pool and project
            pooled = perception_tokens.mean(dim=1)  # (B, D)
            action_tokens = self.pool_proj(pooled).view(B, self.num_actions, self.embed_dim)
            self.last_attention = None
        
        # Stage 3: Decode to logits
        logits, magnitudes = self.decoder(action_tokens)
        self.last_magnitudes = magnitudes.detach()
        
        # Value from pooled perception
        value = self.value_head(perception_tokens.mean(dim=1)).squeeze(-1)
        
        return logits, value, magnitudes
    
    def get_action_and_value(self, obs, action=None):
        logits, value, magnitudes = self.forward(obs)
        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        
        if action is None:
            action = dist.sample()
        
        return action, dist.log_prob(action), value, dist.entropy(), magnitudes, action
    
    def get_value(self, obs):
        _, value, _ = self.forward(obs)
        return value
    
    def contrastive_loss(self, magnitudes: torch.Tensor, chosen_actions: torch.Tensor, 
                         margin: float = 0.3) -> torch.Tensor:
        """
        Push chosen action magnitude higher than others.
        """
        if not self.use_contrastive:
            return torch.tensor(0.0, device=magnitudes.device)
        
        B = magnitudes.size(0)
        
        # Get chosen action magnitude
        chosen_mag = magnitudes[torch.arange(B), chosen_actions]
        
        # Mean of other magnitudes
        mask = torch.ones_like(magnitudes, dtype=torch.bool)
        mask[torch.arange(B), chosen_actions] = False
        other_mag = magnitudes[mask].view(B, -1).mean(dim=1)
        
        # Margin loss
        loss = F.relu(margin - (chosen_mag - other_mag)).mean()
        
        return loss
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    def get_attention_weights(self) -> Optional[np.ndarray]:
        if self.last_attention is not None:
            return self.last_attention.mean(dim=0).cpu().numpy()
        return None


# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def collect_rollout(env, policy, num_steps: int, device: str):
    """Collect experience for PPO update."""
    obs_list, actions_list, log_probs_list = [], [], []
    rewards_list, dones_list, values_list = [], [], []
    magnitudes_list, chosen_list = [], []
    
    obs = env.reset()
    policy.eval()
    
    with torch.no_grad():
        for _ in range(num_steps):
            obs_t = torch.FloatTensor(obs).to(device)
            action, log_prob, value, entropy, magnitudes, chosen = policy.get_action_and_value(obs_t)
            
            obs_list.append(obs.copy())
            actions_list.append(action.cpu().numpy())
            log_probs_list.append(log_prob.cpu().numpy())
            values_list.append(value.cpu().numpy())
            
            if magnitudes is not None:
                magnitudes_list.append(magnitudes.mean(dim=0).cpu().numpy())
                chosen_list.append(chosen.cpu().numpy())
            
            next_obs, rewards, dones, _ = env.step(action.cpu().numpy())
            rewards_list.append(rewards)
            dones_list.append(dones.astype(np.float32))
            obs = next_obs
        
        obs_t = torch.FloatTensor(obs).to(device)
        next_value = policy.get_value(obs_t).cpu().numpy()
    
    policy.train()
    
    # Compute GAE
    obs_arr = np.array(obs_list)
    rewards_arr = np.array(rewards_list)
    dones_arr = np.array(dones_list)
    values_arr = np.array(values_list)
    
    advantages = np.zeros_like(rewards_arr)
    lastgae = 0
    for t in reversed(range(num_steps)):
        if t == num_steps - 1:
            next_val = next_value
        else:
            next_val = values_arr[t + 1]
        delta = rewards_arr[t] + config.gamma * next_val * (1 - dones_arr[t]) - values_arr[t]
        advantages[t] = lastgae = delta + config.gamma * config.gae_lambda * (1 - dones_arr[t]) * lastgae
    
    returns = advantages + values_arr
    
    # Episode stats
    episode_rewards = []
    current_rewards = np.zeros(env.num_envs)
    for t in range(num_steps):
        current_rewards += rewards_arr[t]
        for i, done in enumerate(dones_arr[t]):
            if done:
                episode_rewards.append(current_rewards[i])
                current_rewards[i] = 0
    
    return {
        'obs': torch.FloatTensor(obs_arr.reshape(-1, *obs_arr.shape[2:])),
        'actions': torch.LongTensor(np.array(actions_list).reshape(-1)),
        'log_probs': torch.FloatTensor(np.array(log_probs_list).reshape(-1)),
        'advantages': torch.FloatTensor(advantages.reshape(-1)),
        'returns': torch.FloatTensor(returns.reshape(-1)),
        'mean_reward': np.mean(episode_rewards) if episode_rewards else 0,
        'num_episodes': len(episode_rewards),
        'magnitudes': magnitudes_list
    }


def ppo_update(policy, rollout, optimizer, config, is_trn: bool = False):
    """PPO update with optional contrastive loss for TRN."""
    device = config.device
    obs = rollout['obs'].to(device)
    actions = rollout['actions'].to(device)
    old_log_probs = rollout['log_probs'].to(device)
    advantages = rollout['advantages'].to(device)
    returns = rollout['returns'].to(device)
    
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    batch_size = obs.size(0)
    minibatch_size = batch_size // 4
    
    total_loss = 0
    total_contrastive = 0
    num_updates = 0
    
    for _ in range(config.ppo_epochs):
        indices = torch.randperm(batch_size)
        for start in range(0, batch_size, minibatch_size):
            end = start + minibatch_size
            mb_idx = indices[start:end]
            
            mb_obs = obs[mb_idx]
            mb_actions = actions[mb_idx]
            mb_old_lp = old_log_probs[mb_idx]
            mb_adv = advantages[mb_idx]
            mb_ret = returns[mb_idx]
            
            _, new_lp, new_val, entropy, magnitudes, _ = policy.get_action_and_value(
                mb_obs, mb_actions
            )
            
            # PPO loss
            ratio = torch.exp(new_lp - mb_old_lp)
            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1 - config.clip_ratio, 1 + config.clip_ratio) * mb_adv
            policy_loss = -torch.min(surr1, surr2).mean()
            
            value_loss = F.mse_loss(new_val, mb_ret)
            entropy_loss = -entropy.mean()
            
            # Contrastive loss for TRN
            contrastive = torch.tensor(0.0, device=device)
            if is_trn and magnitudes is not None and hasattr(policy, 'contrastive_loss'):
                contrastive = policy.contrastive_loss(magnitudes, mb_actions, config.contrastive_margin)
            
            loss = (policy_loss + config.value_coef * value_loss + 
                    config.entropy_coef * entropy_loss + 
                    config.contrastive_weight * contrastive)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), config.max_grad_norm)
            optimizer.step()
            
            total_loss += loss.item()
            total_contrastive += contrastive.item()
            num_updates += 1
    
    return total_loss / num_updates, total_contrastive / num_updates


def train_policy(policy, game: str, config: Config, num_steps: int, name: str, 
                 is_trn: bool = False, save_dir: str = None) -> Dict:
    """Train a policy and return results + visualizations."""
    policy.to(config.device)
    optimizer = optim.Adam(policy.parameters(), lr=config.learning_rate)
    
    history = {'reward': [], 'loss': [], 'contrastive': []}
    magnitudes_history = []
    attention_samples = []
    perception_samples = []
    
    params = policy.count_parameters()
    print(f"    Training {name} | Parameters: {params:,}")
    
    env = VecEnv(game, config.num_envs)
    
    try:
        for step in range(num_steps):
            rollout = collect_rollout(env, policy, config.num_steps_per_rollout, config.device)
            loss, contrastive = ppo_update(policy, rollout, optimizer, config, is_trn)
            
            history['reward'].append(rollout['mean_reward'])
            history['loss'].append(loss)
            history['contrastive'].append(contrastive)
            
            # Collect visualization data
            if rollout['magnitudes']:
                magnitudes_history.append(np.mean(rollout['magnitudes'], axis=0))
            
            if is_trn and step % 50 == 0:
                attn = policy.get_attention_weights()
                if attn is not None:
                    attention_samples.append(attn)
                if policy.last_perception_tokens is not None:
                    perception_samples.append(policy.last_perception_tokens[:5].cpu().numpy())
            
            if (step + 1) % 30 == 0:
                recent = np.mean(history['reward'][-30:])
                msg = f"      Step {step+1:4d}/{num_steps} | Reward: {rollout['mean_reward']:6.2f} | Recent: {recent:6.2f}"
                if is_trn:
                    msg += f" | Contrastive: {contrastive:.4f}"
                print(msg)
    finally:
        env.close()
    
    # Save visualizations immediately
    if save_dir:
        # Learning curve
        save_learning_curve(
            history['reward'], name,
            os.path.join(save_dir, f'{name.lower().replace(" ", "_")}_learning.png')
        )
        
        # TRN-specific visualizations
        if is_trn:
            if magnitudes_history:
                save_action_token_magnitudes(
                    magnitudes_history,
                    os.path.join(save_dir, f'{name.lower().replace(" ", "_")}_magnitudes.png')
                )
            
            if attention_samples:
                save_attention_heatmap(
                    attention_samples[-1],  # Last attention
                    os.path.join(save_dir, f'{name.lower().replace(" ", "_")}_attention.png')
                )
            
            if perception_samples:
                tokens = np.concatenate(perception_samples, axis=0)
                save_perception_tokens_pca(
                    tokens,
                    os.path.join(save_dir, f'{name.lower().replace(" ", "_")}_tokens_pca.png')
                )
    
    return {
        'history': history,
        'params': params,
        'final_reward': np.mean(history['reward'][-30:]) if len(history['reward']) >= 30 else np.mean(history['reward']),
        'magnitudes': magnitudes_history,
        'attention': attention_samples
    }


def evaluate_policy(policy, game: str, config: Config, num_episodes: int = 20, 
                    is_trn: bool = False) -> Dict:
    """Evaluate policy performance."""
    policy.eval()
    rewards = []
    
    with torch.no_grad():
        for _ in range(num_episodes):
            if MINATAR_AVAILABLE:
                env = MinAtarEnv(game)
            else:
                env = FallbackEnv()
            
            obs = env.reset()
            total = 0
            done = False
            steps = 0
            
            while not done and steps < 1000:
                obs_t = torch.FloatTensor(obs).unsqueeze(0).to(config.device)
                logits, _, _ = policy.forward(obs_t) if is_trn else (policy.forward(obs_t)[0], None, None)
                action = torch.argmax(logits, dim=-1).item()
                obs, r, done, _ = env.step(action)
                total += r
                steps += 1
            
            rewards.append(total)
            env.close()
    
    policy.train()
    return {'mean': np.mean(rewards), 'std': np.std(rewards), 'all': rewards}


# ═══════════════════════════════════════════════════════════════════════════════
# STATISTICAL ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

def statistical_test(group1: List[float], group2: List[float], name1: str, name2: str) -> Dict:
    """Perform statistical comparison between two groups."""
    g1, g2 = np.array(group1), np.array(group2)
    
    # Welch's t-test
    t_stat, t_p = stats.ttest_ind(g1, g2, equal_var=False)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((np.var(g1) + np.var(g2)) / 2)
    cohens_d = (np.mean(g1) - np.mean(g2)) / (pooled_std + 1e-8)
    
    # Effect interpretation
    if abs(cohens_d) >= 0.8:
        effect = 'large'
    elif abs(cohens_d) >= 0.5:
        effect = 'medium'
    elif abs(cohens_d) >= 0.2:
        effect = 'small'
    else:
        effect = 'negligible'
    
    return {
        f'{name1}_mean': np.mean(g1),
        f'{name1}_std': np.std(g1),
        f'{name2}_mean': np.mean(g2),
        f'{name2}_std': np.std(g2),
        'difference': np.mean(g1) - np.mean(g2),
        'improvement_pct': (np.mean(g1) - np.mean(g2)) / (abs(np.mean(g2)) + 1e-8) * 100,
        'p_value': t_p,
        'cohens_d': cohens_d,
        'effect_size': effect,
        'significant': t_p < 0.05
    }


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_seed_experiment(seed: int, config: Config) -> Dict:
    """Run complete experiment for one seed."""
    set_seed(seed)
    results = {}
    seed_dir = os.path.join(config.save_dir, f'seed_{seed}')
    os.makedirs(seed_dir, exist_ok=True)
    
    print_subsection(f"Seed {seed}")
    
    # ═══════════════════════════════════════════════════════════════════════════
    # BASELINE CNN-PPO
    # ═══════════════════════════════════════════════════════════════════════════
    print("  [1/4] Baseline CNN-PPO")
    baseline = BaselineCNN(config.num_actions, config.embed_dim)
    baseline_result = train_policy(
        baseline, config.game, config, config.num_train_steps, 
        "Baseline", is_trn=False, save_dir=seed_dir
    )
    baseline_eval = evaluate_policy(baseline, config.game, config, is_trn=False)
    
    results['baseline'] = {
        'train_reward': baseline_result['final_reward'],
        'eval_mean': baseline_eval['mean'],
        'eval_std': baseline_eval['std'],
        'eval_all': baseline_eval['all'],
        'params': baseline_result['params'],
        'history': baseline_result['history']['reward']
    }
    print(f"    Train: {results['baseline']['train_reward']:.2f} | Eval: {baseline_eval['mean']:.2f} ± {baseline_eval['std']:.2f}")
    
    # ═══════════════════════════════════════════════════════════════════════════
    # TRN FULL MODEL
    # ═══════════════════════════════════════════════════════════════════════════
    print("  [2/4] TRN (Full Model)")
    set_seed(seed)
    trn = TokenizedReasoningNetwork(
        config.num_actions, config.embed_dim,
        use_cross_attention=True,
        use_positional=True,
        use_contrastive=True,
        use_bias=True
    )
    trn_result = train_policy(
        trn, config.game, config, config.num_train_steps,
        "TRN_Full", is_trn=True, save_dir=seed_dir
    )
    trn_eval = evaluate_policy(trn, config.game, config, is_trn=True)
    
    results['trn'] = {
        'train_reward': trn_result['final_reward'],
        'eval_mean': trn_eval['mean'],
        'eval_std': trn_eval['std'],
        'eval_all': trn_eval['all'],
        'params': trn_result['params'],
        'history': trn_result['history']['reward'],
        'attention': trn_result['attention'],
        'magnitudes': trn_result['magnitudes']
    }
    print(f"    Train: {results['trn']['train_reward']:.2f} | Eval: {trn_eval['mean']:.2f} ± {trn_eval['std']:.2f}")
    
    # Save comparison learning curve
    save_learning_curve(
        trn_result['history']['reward'], 'TRN',
        os.path.join(seed_dir, 'comparison_learning.png'),
        baseline_rewards=baseline_result['history']['reward']
    )
    
    # ═══════════════════════════════════════════════════════════════════════════
    # DIMENSION ABLATION
    # ═══════════════════════════════════════════════════════════════════════════
    print("  [3/4] Dimension Ablation")
    dim_results = {}
    
    for dim in [32, 64, 128]:
        set_seed(seed)
        model = TokenizedReasoningNetwork(config.num_actions, dim)
        result = train_policy(
            model, config.game, config, config.num_train_steps // 2,
            f"TRN_dim{dim}", is_trn=True, save_dir=None
        )
        eval_result = evaluate_policy(model, config.game, config, num_episodes=10, is_trn=True)
        
        dim_results[dim] = {
            'mean': eval_result['mean'],
            'std': eval_result['std'],
            'params': result['params']
        }
        print(f"      dim={dim}: {eval_result['mean']:.2f} ± {eval_result['std']:.2f} | Params: {result['params']:,}")
    
    results['dim_ablation'] = dim_results
    save_dimension_comparison(dim_results, os.path.join(seed_dir, 'dimension_ablation.png'))
    
    # ═══════════════════════════════════════════════════════════════════════════
    # COMPONENT ABLATION
    # ═══════════════════════════════════════════════════════════════════════════
    print("  [4/4] Component Ablation")
    ablation_configs = {
        'Full TRN': {'use_cross_attention': True, 'use_positional': True, 
                     'use_contrastive': True, 'use_bias': True},
        'No Cross-Attn': {'use_cross_attention': False, 'use_positional': True,
                          'use_contrastive': True, 'use_bias': True},
        'No Position': {'use_cross_attention': True, 'use_positional': False,
                        'use_contrastive': True, 'use_bias': True},
        'No Contrastive': {'use_cross_attention': True, 'use_positional': True,
                           'use_contrastive': False, 'use_bias': True},
        'No Bias': {'use_cross_attention': True, 'use_positional': True,
                    'use_contrastive': True, 'use_bias': False},
    }
    
    ablation_results = {}
    for name, cfg in ablation_configs.items():
        set_seed(seed)
        model = TokenizedReasoningNetwork(config.num_actions, config.embed_dim, **cfg)
        result = train_policy(
            model, config.game, config, config.num_train_steps // 2,
            name, is_trn=True, save_dir=None
        )
        eval_result = evaluate_policy(model, config.game, config, num_episodes=10, is_trn=True)
        
        ablation_results[name] = {
            'mean': eval_result['mean'],
            'std': eval_result['std'],
            'params': result['params']
        }
        print(f"      {name}: {eval_result['mean']:.2f}")
    
    results['ablation'] = ablation_results
    save_ablation_chart(ablation_results, os.path.join(seed_dir, 'component_ablation.png'))
    
    return results


def main():
    start_time = time.time()
    
    print_section("TOKENIZED REASONING NETWORK (TRN)")
    print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║ NOVEL ARCHITECTURE                                                           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Stage 1: Perception Tokenizer                                                ║
║   - CNN extracts features → 5×5 spatial tokens (25 tokens)                  ║
║   - 2D positional embeddings encode spatial location                        ║
║                                                                              ║
║ Stage 2: Cross-Attention Translator                                         ║
║   - 6 learnable action queries (one per action)                             ║
║   - Each action ATTENDS to perception tokens                                ║
║   - Output: 6 action tokens (what each action "knows")                      ║
║                                                                              ║
║ Stage 3: Action Decoder                                                      ║
║   - Magnitude of action tokens = activation level                           ║
║   - Learnable bias for baseline preferences                                 ║
║   - Contrastive loss strengthens chosen action                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

Configuration:
  Game: {config.game} (single game for clean validation)
  Seeds: {config.seeds}
  Training Steps: {config.num_train_steps}
  Embedding Dim: {config.embed_dim}
  Device: {config.device}
    """)
    
    all_results = []
    
    for seed in config.seeds:
        print_section(f"SEED {seed}")
        try:
            results = run_seed_experiment(seed, config)
            all_results.append(results)
            print(f"\n  Seed {seed} completed")
        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback
            traceback.print_exc()
    
    if not all_results:
        print("No successful experiments")
        return
    
    # ═══════════════════════════════════════════════════════════════════════════
    # AGGREGATE RESULTS
    # ═══════════════════════════════════════════════════════════════════════════
    print_section("AGGREGATE RESULTS")
    
    baseline_evals = [r['baseline']['eval_mean'] for r in all_results]
    trn_evals = [r['trn']['eval_mean'] for r in all_results]
    
    print_subsection("Performance Summary")
    print(f"""
  ┌────────────────────────────────────────────────────────────────────────┐
  │ METHOD            │ EVAL REWARD          │ PARAMETERS                 │
  ├────────────────────────────────────────────────────────────────────────┤
  │ Baseline CNN      │ {np.mean(baseline_evals):5.2f} ± {np.std(baseline_evals):4.2f}         │ {all_results[0]['baseline']['params']:,}                     │
  │ TRN (Ours)        │ {np.mean(trn_evals):5.2f} ± {np.std(trn_evals):4.2f}         │ {all_results[0]['trn']['params']:,}                     │
  └────────────────────────────────────────────────────────────────────────┘
    """)
    
    # Statistical test
    print_subsection("Statistical Analysis")
    comparison = statistical_test(trn_evals, baseline_evals, 'TRN', 'Baseline')
    print(f"""
  TRN vs Baseline:
    TRN:      {comparison['TRN_mean']:.2f} ± {comparison['TRN_std']:.2f}
    Baseline: {comparison['Baseline_mean']:.2f} ± {comparison['Baseline_std']:.2f}
    
    Difference:    {comparison['difference']:+.2f}
    Improvement:   {comparison['improvement_pct']:+.1f}%
    p-value:       {comparison['p_value']:.6f}
    Cohen's d:     {comparison['cohens_d']:.3f} ({comparison['effect_size']})
    Significant:   {comparison['significant']}
    """)
    
    # Dimension ablation aggregate
    print_subsection("Dimension Ablation (Aggregated)")
    print("\n  | Dimension | Mean Reward | Std |")
    print("  |-----------|-------------|-----|")
    for dim in [32, 64, 128]:
        means = [r['dim_ablation'][dim]['mean'] for r in all_results]
        print(f"  | {dim:9d} | {np.mean(means):11.2f} | {np.std(means):.2f} |")
    
    # Component ablation aggregate
    print_subsection("Component Ablation (Aggregated)")
    full_means = [r['ablation']['Full TRN']['mean'] for r in all_results]
    full_mean = np.mean(full_means)
    
    print("\n  | Component Removed | Mean Reward | Δ from Full |")
    print("  |-------------------|-------------|-------------|")
    print(f"  | Full TRN          | {full_mean:11.2f} |      -      |")
    
    for name in ['No Cross-Attn', 'No Position', 'No Contrastive', 'No Bias']:
        means = [r['ablation'][name]['mean'] for r in all_results]
        mean = np.mean(means)
        delta = full_mean - mean
        print(f"  | {name:17s} | {mean:11.2f} | {delta:+10.2f} |")
    
    # Save final comparison
    print_subsection("Saving Final Visualizations")
    
    final_results = {
        'Baseline CNN': {'mean': np.mean(baseline_evals), 'std': np.std(baseline_evals)},
        'TRN (Ours)': {'mean': np.mean(trn_evals), 'std': np.std(trn_evals)}
    }
    save_comparison_bar(final_results, os.path.join(config.save_dir, 'final_comparison.png'))
    
    # Aggregate learning curves
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i, r in enumerate(all_results):
        alpha = 0.3
        ax.plot(r['baseline']['history'], 'r-', alpha=alpha, linewidth=1)
        ax.plot(r['trn']['history'], 'b-', alpha=alpha, linewidth=1)
    
    # Mean curves
    min_len = min(len(r['baseline']['history']) for r in all_results)
    baseline_mean = np.mean([r['baseline']['history'][:min_len] for r in all_results], axis=0)
    trn_mean = np.mean([r['trn']['history'][:min_len] for r in all_results], axis=0)
    
    ax.plot(baseline_mean, 'r-', linewidth=3, label='Baseline (mean)')
    ax.plot(trn_mean, 'b-', linewidth=3, label='TRN (mean)')
    
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Mean Reward')
    ax.set_title('Learning Curves Across Seeds')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.save_dir, 'aggregate_learning.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {config.save_dir}/aggregate_learning.png")
    
    # ═══════════════════════════════════════════════════════════════════════════
    # PUBLICATION TABLES
    # ═══════════════════════════════════════════════════════════════════════════
    print_section("PUBLICATION TABLES")
    
    print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║ TABLE 1: Main Results                                                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Method          │ Reward (mean±std) │ Parameters │ p-value  │ Effect Size   ║
╠══════════════════════════════════════════════════════════════════════════════╣""")
    print(f"║ Baseline CNN    │ {np.mean(baseline_evals):5.2f} ± {np.std(baseline_evals):4.2f}       │ {all_results[0]['baseline']['params']:10,} │    -     │      -        ║")
    print(f"║ TRN (Ours)      │ {np.mean(trn_evals):5.2f} ± {np.std(trn_evals):4.2f}       │ {all_results[0]['trn']['params']:10,} │ {comparison['p_value']:.4f}   │ {comparison['cohens_d']:+.2f} ({comparison['effect_size']:s})  ║")
    print("╚══════════════════════════════════════════════════════════════════════════════╝")
    
    print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║ TABLE 2: Component Ablation                                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Configuration     │ Mean Reward │ Contribution (Δ)                          ║
╠══════════════════════════════════════════════════════════════════════════════╣""")
    print(f"║ Full TRN          │ {full_mean:11.2f} │           -                              ║")
    for name in ['No Cross-Attn', 'No Position', 'No Contrastive', 'No Bias']:
        means = [r['ablation'][name]['mean'] for r in all_results]
        mean = np.mean(means)
        delta = full_mean - mean
        print(f"║ {name:17s} │ {mean:11.2f} │ {delta:+10.2f}                              ║")
    print("╚══════════════════════════════════════════════════════════════════════════════╝")
    
    # ═══════════════════════════════════════════════════════════════════════════
    # FINAL SUMMARY
    # ═══════════════════════════════════════════════════════════════════════════
    print_section("EXPERIMENT SUMMARY")
    
    total_time = time.time() - start_time
    
    print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║ TOKENIZED REASONING NETWORK - RESULTS                                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║ KEY FINDINGS:                                                                ║
║                                                                              ║
║   1. Performance:                                                            ║
║      TRN:      {np.mean(trn_evals):5.2f} ± {np.std(trn_evals):4.2f}                                               ║
║      Baseline: {np.mean(baseline_evals):5.2f} ± {np.std(baseline_evals):4.2f}                                               ║
║      Improvement: {comparison['improvement_pct']:+.1f}%                                                ║
║                                                                              ║
║   2. Statistical Significance:                                               ║
║      p-value: {comparison['p_value']:.6f}                                                     ║
║      Effect size: {comparison['cohens_d']:.2f} ({comparison['effect_size']})                                          ║
║                                                                              ║
║   3. Most Important Components (by ablation contribution):                   ║""")

    # Sort by contribution
    contributions = {}
    for name in ['No Cross-Attn', 'No Position', 'No Contrastive', 'No Bias']:
        means = [r['ablation'][name]['mean'] for r in all_results]
        contributions[name] = full_mean - np.mean(means)
    
    sorted_contribs = sorted(contributions.items(), key=lambda x: x[1], reverse=True)
    for i, (name, contrib) in enumerate(sorted_contribs, 1):
        component = name.replace('No ', '')
        print(f"║      {i}. {component:15s} (Δ = {contrib:+.2f})                                      ║")

    print(f"""║                                                                              ║
║   4. Visualizations saved to: {config.save_dir}                           ║
║                                                                              ║
║   5. Computation:                                                            ║
║      Seeds: {len(all_results)}/{len(config.seeds)} successful                                              ║
║      Total time: {total_time:.1f}s                                                      ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

INTERPRETABILITY HIGHLIGHTS:
  - Cross-attention maps show WHERE each action "looks" in the scene
  - Action token magnitudes show HOW activated each action is
  - Perception tokens cluster by spatial position (see PCA plots)
  - All visualizations saved for paper figures
    """)
    
    print("=" * 80)
    print(" EXPERIMENT COMPLETED")
    print("=" * 80)

if __name__ == "__main__":
    main()

 TOKENIZED REASONING NETWORK (TRN) - INSTALLATION
MinAtar: Available
PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla P100-PCIE-16GB
Memory: 17.1 GB

 TOKENIZED REASONING NETWORK (TRN)

╔══════════════════════════════════════════════════════════════════════════════╗
║ NOVEL ARCHITECTURE                                                           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ Stage 1: Perception Tokenizer                                                ║
║   - CNN extracts features → 5×5 spatial tokens (25 tokens)                  ║
║   - 2D positional embeddings encode spatial location                        ║
║                                                                              ║
║ Stage 2: Cross-Attention Translator                                         ║
║   - 6 learnable action queries (one per action)                             ║
║   - Each action ATTENDS to perception tokens                                ║
║   - Output: 6

In [2]:
!zip -r trn_figures.zip /kaggle/working/trn_figures

  adding: kaggle/working/trn_figures/ (stored 0%)
  adding: kaggle/working/trn_figures/seed_456/ (stored 0%)
  adding: kaggle/working/trn_figures/seed_456/trn_full_magnitudes.png (deflated 7%)
  adding: kaggle/working/trn_figures/seed_456/trn_full_tokens_pca.png (deflated 8%)
  adding: kaggle/working/trn_figures/seed_456/component_ablation.png (deflated 33%)
  adding: kaggle/working/trn_figures/seed_456/baseline_learning.png (deflated 11%)
  adding: kaggle/working/trn_figures/seed_456/comparison_learning.png (deflated 7%)
  adding: kaggle/working/trn_figures/seed_456/trn_full_learning.png (deflated 11%)
  adding: kaggle/working/trn_figures/seed_456/dimension_ablation.png (deflated 24%)
  adding: kaggle/working/trn_figures/seed_456/trn_full_attention.png (deflated 30%)
  adding: kaggle/working/trn_figures/seed_42/ (stored 0%)
  adding: kaggle/working/trn_figures/seed_42/trn_full_magnitudes.png (deflated 7%)
  adding: kaggle/working/trn_figures/seed_42/trn_full_tokens_pca.png (deflated 7